In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [ ]:
from pathlib import Path
import yaml

# ============================================================
# VisDrone_Final — Dataset YAML
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final"
)

YAML_PATH = Path("/kaggle/working/visdrone_final.yaml")

# Check dataset
print("Dataset exists:", DATASET_ROOT.exists())
print("Dataset path  :", DATASET_ROOT)

# Check required directories
required_dirs = [
    DATASET_ROOT / "train" / "images",
    DATASET_ROOT / "train" / "labels",
    DATASET_ROOT / "val" / "images",
    DATASET_ROOT / "val" / "labels",
    DATASET_ROOT / "test" / "images",
    DATASET_ROOT / "test" / "labels",
]

for d in required_dirs:
    print(f"{d}: {d.exists()}")

# ============================================================
# YAML configuration
# ============================================================

data = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",

    "nc": 10,

    "names": [
        "pedestrian",
        "people",
        "bicycle",
        "car",
        "van",
        "truck",
        "tricycle",
        "awning-tricycle",
        "bus",
        "motor"
    ]
}

# Write YAML
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("\n" + "=" * 60)
print("YAML CREATED")
print("=" * 60)

print(YAML_PATH)
print("\nContents:\n")

with open(YAML_PATH, "r") as f:
    print(f.read())

In [ ]:
# ============================================================
# CELL 1 — CLEAN ENVIRONMENT SETUP
# ============================================================

import os
import sys
import torch
import ultralytics

from ultralytics import YOLO

print("=" * 70)
print("CLEAN YOLOv8n + CBAM EXPERIMENT")
print("=" * 70)

print(f"Python       : {sys.version.split()[0]}")
print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version : {torch.version.cuda}")
    print(
        f"GPU memory   : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Seed         : {SEED}")

# Dataset
DATA_YAML = "/kaggle/working/visdrone_final.yaml"

print("\nDataset YAML:", DATA_YAML)
print("Exists      :", os.path.exists(DATA_YAML))

assert torch.cuda.is_available(), "CUDA GPU is not available."
assert os.path.exists(DATA_YAML), "VisDrone YAML file not found."

print("\n✓ Environment check passed.")

In [3]:
# ============================================================
# CELL 2 — CORRECTED CBAM MODULE + SAFE PARSER
# ============================================================

import torch
import torch.nn as nn

from ultralytics.nn import tasks
from ultralytics.nn.modules import C2f


# ============================================================
# 0. SAVE THE ORIGINAL ULTRALYTICS PARSER
# ============================================================

# IMPORTANT:
# This must be captured BEFORE installing our CBAM parser hook.

ORIGINAL_PARSE_MODEL = tasks.parse_model


# ============================================================
# 1. CBAM
# ============================================================

class ChannelAttention(nn.Module):

    def __init__(self, channels, reduction=16):
        super().__init__()

        hidden = max(channels // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))

        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):

    def __init__(self, kernel_size=7):
        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        max_out, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        return self.sigmoid(self.conv(x))


class CBAM(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16,
        kernel_size=7
    ):
        super().__init__()

        self.channel_attention = ChannelAttention(
            channels,
            reduction
        )

        self.spatial_attention = SpatialAttention(
            kernel_size
        )

    def forward(self, x):

        x = x * self.channel_attention(x)
        x = x * self.spatial_attention(x)

        return x


# ============================================================
# 2. C2f + CBAM WRAPPER
# ============================================================

class C2fCBAM(nn.Module):

    def __init__(
        self,
        original_c2f,
        channels,
        reduction=16,
        kernel_size=7
    ):
        super().__init__()

        # Preserve original pretrained C2f
        self.c2f = original_c2f

        # CBAM added AFTER C2f
        self.cbam = CBAM(
            channels,
            reduction=reduction,
            kernel_size=kernel_size
        )

        # Preserve Ultralytics metadata
        self.f = getattr(
            original_c2f,
            "f",
            -1
        )

        self.i = getattr(
            original_c2f,
            "i",
            -1
        )

        self.type = "C2fCBAM"

        self.np = sum(
            p.numel()
            for p in self.parameters()
        )

    def forward(self, x):

        x = self.c2f(x)
        x = self.cbam(x)

        return x


# ============================================================
# 3. SAFE CBAM PARSER
# ============================================================

def cbam_parse_model(d, ch, verbose=True):

    # --------------------------------------------------------
    # Ordinary YOLO model
    # --------------------------------------------------------

    if not d.get("cbam", False):

        return ORIGINAL_PARSE_MODEL(
            d,
            ch,
            verbose
        )

    # --------------------------------------------------------
    # Copy YAML
    # --------------------------------------------------------

    d = dict(d)

    backbone = []

    # --------------------------------------------------------
    # Convert C2fCBAM targets to ordinary C2f
    #
    # The ORIGINAL parser constructs the standard YOLOv8n
    # architecture first.
    # --------------------------------------------------------

    for layer_idx, layer in enumerate(d["backbone"]):

        layer = list(layer)

        module = layer[2]

        if (
            layer_idx in {4, 6, 8}
            and (
                module == "C2fCBAM"
                or getattr(module, "__name__", "") == "C2fCBAM"
            )
        ):
            layer[2] = "C2f"

        backbone.append(layer)

    d["backbone"] = backbone

    # --------------------------------------------------------
    # Build standard YOLOv8n
    # --------------------------------------------------------

    model, save = ORIGINAL_PARSE_MODEL(
        d,
        ch,
        verbose
    )

    # --------------------------------------------------------
    # Insert CBAM after C2f layers 4, 6, 8
    # --------------------------------------------------------

    target_channels = {
        4: 64,
        6: 128,
        8: 256
    }

    for idx, channels in target_channels.items():

        original_block = model[idx]

        wrapped_block = C2fCBAM(
            original_c2f=original_block,
            channels=channels
        )

        # Preserve metadata
        wrapped_block.f = getattr(
            original_block,
            "f",
            -1
        )

        wrapped_block.i = getattr(
            original_block,
            "i",
            idx
        )

        model[idx] = wrapped_block

    return model, save


# ============================================================
# 4. INSTALL EXACTLY ONE PARSER HOOK
# ============================================================

tasks.parse_model = cbam_parse_model


# ============================================================
# 5. VERIFICATION
# ============================================================

print("=" * 70)
print("CORRECTED CBAM PARSER")
print("=" * 70)

print("✓ Original Ultralytics parser captured")
print("✓ CBAM module defined")
print("✓ C2fCBAM wrapper defined")
print("✓ Standard C2f layers will be built first")
print("✓ CBAM inserted after layers 4, 6, 8")
print("✓ No importlib.reload()")
print("✓ No global DetectionModel replacement")
print("✓ Exactly one parser hook installed")

print("\nTarget:")
print("  Layer 4 → 64 channels  → C2fCBAM")
print("  Layer 6 → 128 channels → C2fCBAM")
print("  Layer 8 → 256 channels → C2fCBAM")

print("\n✓ Corrected Cell 2 completed.")

CORRECTED CBAM PARSER
✓ Original Ultralytics parser captured
✓ CBAM module defined
✓ C2fCBAM wrapper defined
✓ Standard C2f layers will be built first
✓ CBAM inserted after layers 4, 6, 8
✓ No importlib.reload()
✓ No global DetectionModel replacement
✓ Exactly one parser hook installed

Target:
  Layer 4 → 64 channels  → C2fCBAM
  Layer 6 → 128 channels → C2fCBAM
  Layer 8 → 256 channels → C2fCBAM

✓ Corrected Cell 2 completed.


In [5]:
# ============================================================
# CELL 3 — CUSTOM YOLOv8n-CBAM YAML
# ============================================================

import yaml
from pathlib import Path

CBAM_YAML = "/kaggle/working/yolov8n_cbam.yaml"

cbam_yaml = {
    "nc": 10,
    "cbam": True,

    "scales": {
        "n": [0.33, 0.25, 1024]
    },

    "backbone": [
        [-1, 1, "Conv", [64, 3, 2]],
        [-1, 1, "Conv", [128, 3, 2]],
        [-1, 3, "C2f", [128, True]],
        [-1, 1, "Conv", [256, 3, 2]],

        # CBAM target
        [-1, 6, "C2fCBAM", [256, True]],

        [-1, 1, "Conv", [512, 3, 2]],

        # CBAM target
        [-1, 6, "C2fCBAM", [512, True]],

        [-1, 1, "Conv", [1024, 3, 2]],

        # CBAM target
        [-1, 3, "C2fCBAM", [1024, True]],

        [-1, 1, "SPPF", [1024, 5]],
    ],

    "head": [
        [-1, 1, "nn.Upsample", [None, 2, "nearest"]],
        [[-1, 6], 1, "Concat", [1]],
        [-1, 3, "C2f", [512]],

        [-1, 1, "nn.Upsample", [None, 2, "nearest"]],
        [[-1, 4], 1, "Concat", [1]],
        [-1, 3, "C2f", [256]],

        [-1, 1, "Conv", [256, 3, 2]],
        [[-1, 12], 1, "Concat", [1]],
        [-1, 3, "C2f", [512]],

        [-1, 1, "Conv", [512, 3, 2]],
        [[-1, 9], 1, "Concat", [1]],
        [-1, 3, "C2f", [1024]],

        [[15, 18, 21], 1, "Detect", [10]],
    ],
}

with open(CBAM_YAML, "w") as f:
    yaml.safe_dump(
        cbam_yaml,
        f,
        sort_keys=False
    )

print("=" * 70)
print("CUSTOM YOLOv8n-CBAM YAML")
print("=" * 70)

print("File   :", CBAM_YAML)
print("Exists :", Path(CBAM_YAML).exists())
print("CBAM   :", cbam_yaml["cbam"])
print("Classes:", cbam_yaml["nc"])

print("\n✓ Cell 3 completed.")

CUSTOM YOLOv8n-CBAM YAML
File   : /kaggle/working/yolov8n_cbam.yaml
Exists : True
CBAM   : True
Classes: 10

✓ Cell 3 completed.


In [ ]:
# ============================================================
# CELL 4 — BUILD AND VERIFY YOLOv8n-CBAM
# ============================================================

from ultralytics import YOLO
import torch

print("=" * 70)
print("BUILDING YOLOv8n + CBAM")
print("=" * 70)

# Build directly from the custom YAML.
# Do NOT load yolov8n.pt yet.
cbam_model = YOLO(CBAM_YAML)

model = cbam_model.model

# ------------------------------------------------------------
# Count CBAM modules
# ------------------------------------------------------------

cbam_count = sum(
    1 for m in model.modules()
    if m.__class__.__name__ == "CBAM"
)

c2fcbam_count = sum(
    1 for m in model.modules()
    if m.__class__.__name__ == "C2fCBAM"
)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

# ------------------------------------------------------------
# Print architecture
# ------------------------------------------------------------

print("\nCBAM modules      :", cbam_count)
print("C2fCBAM modules   :", c2fcbam_count)

print("\nTotal parameters  :", f"{total_params:,}")
print("Trainable params  :", f"{trainable_params:,}")

print("\nExpected:")
print("CBAM modules      : 3")
print("C2fCBAM modules   : 3")
print("Parameters        : 3,023,844")

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert cbam_count == 3, (
    f"Expected 3 CBAM modules, found {cbam_count}"
)

assert c2fcbam_count == 3, (
    f"Expected 3 C2fCBAM modules, found {c2fcbam_count}"
)

assert total_params == 3_023_844, (
    f"Expected 3,023,844 parameters, found {total_params}"
)

print("\n✓ CBAM architecture is correct.")

In [ ]:
# ============================================================
# CELL 5 — TRANSFER YOLOv8n PRETRAINED WEIGHTS
# ============================================================

from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8n → YOLOv8n-CBAM PRETRAINED WEIGHT TRANSFER")
print("=" * 70)

# ------------------------------------------------------------
# Load official YOLOv8n pretrained model
# ------------------------------------------------------------

pretrained = YOLO("yolov8n.pt")

src_state = pretrained.model.state_dict()
dst_state = cbam_model.model.state_dict()

compatible = {}

# ------------------------------------------------------------
# 1. Direct key/shape matches
# ------------------------------------------------------------

for key, value in src_state.items():

    if (
        key in dst_state
        and dst_state[key].shape == value.shape
    ):
        compatible[key] = value


# ------------------------------------------------------------
# 2. Explicitly map original C2f weights into C2fCBAM
# ------------------------------------------------------------

cbam_layers = [4, 6, 8]

for layer_idx in cbam_layers:

    original_prefix = f"model.{layer_idx}."
    cbam_prefix = f"model.{layer_idx}.c2f."

    for key, value in src_state.items():

        if key.startswith(original_prefix):

            suffix = key[len(original_prefix):]

            new_key = cbam_prefix + suffix

            if (
                new_key in dst_state
                and dst_state[new_key].shape == value.shape
            ):
                compatible[new_key] = value


# ------------------------------------------------------------
# Load compatible weights
# ------------------------------------------------------------

missing, unexpected = cbam_model.model.load_state_dict(
    compatible,
    strict=False
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nSource tensors       :", len(src_state))
print("Target tensors       :", len(dst_state))
print("Transferred tensors  :", len(compatible))
print("Missing tensors      :", len(missing))
print("Unexpected tensors   :", len(unexpected))

# ------------------------------------------------------------
# Identify CBAM-specific parameters
# ------------------------------------------------------------

cbam_missing = [
    key
    for key in missing
    if ".cbam." in key
]

print("\nCBAM-specific tensors not pretrained :", len(cbam_missing))

if cbam_missing:
    print("✓ CBAM parameters remain newly initialized.")

# ------------------------------------------------------------
# Verify architecture did NOT change
# ------------------------------------------------------------

cbam_count = sum(
    1
    for m in cbam_model.model.modules()
    if m.__class__.__name__ == "CBAM"
)

c2fcbam_count = sum(
    1
    for m in cbam_model.model.modules()
    if m.__class__.__name__ == "C2fCBAM"
)

total_params = sum(
    p.numel()
    for p in cbam_model.model.parameters()
)

print("\nArchitecture after weight transfer:")
print("CBAM modules      :", cbam_count)
print("C2fCBAM modules   :", c2fcbam_count)
print("Total parameters  :", f"{total_params:,}")

assert cbam_count == 3
assert c2fcbam_count == 3
assert total_params == 3_023_844

print("\n✓ Pretrained weight transfer completed.")
print("✓ CBAM architecture preserved.")

In [ ]:
import torch

print("=" * 70)
print("YOLOv8n-CBAM FORWARD-PASS VERIFICATION")
print("=" * 70)

# Use the existing cbam_model from the previous cells
model = cbam_model.model
model.eval()

# Dummy input matching YOLO input format
x = torch.randn(1, 3, 640, 640).to(next(model.parameters()).device)

print(f"Input shape : {tuple(x.shape)}")
print(f"Device      : {x.device}")

with torch.no_grad():
    output = model(x)

print("\nForward pass completed successfully.")

# YOLO Detect output can be a tensor or tuple/list depending on mode
if isinstance(output, (tuple, list)):
    print(f"Output type : {type(output).__name__}")
    for i, out in enumerate(output):
        if torch.is_tensor(out):
            print(f"Output[{i}] shape: {tuple(out.shape)}")
else:
    print(f"Output type : {type(output).__name__}")
    if torch.is_tensor(output):
        print(f"Output shape: {tuple(output.shape)}")

# Verify CBAM modules are still present
cbam_count = sum(
    1 for m in model.modules()
    if m.__class__.__name__ == "CBAM"
)

c2fcbam_count = sum(
    1 for m in model.modules()
    if m.__class__.__name__ == "C2fCBAM"
)

print("\nArchitecture verification:")
print(f"CBAM modules    : {cbam_count}")
print(f"C2fCBAM modules : {c2fcbam_count}")
print(f"Parameters      : {sum(p.numel() for p in model.parameters()):,}")

assert cbam_count == 3
assert c2fcbam_count == 3
assert sum(p.numel() for p in model.parameters()) == 3_023_844

print("\n✓ Forward pass successful.")
print("✓ CBAM modules preserved.")
print("✓ Parameter count unchanged.")
print("✓ Ready for training smoke test.")

In [ ]:
from ultralytics import YOLO
import torch
from pathlib import Path

print("=" * 70)
print("YOLOv8n-CBAM — 1-EPOCH TRAINING SMOKE TEST")
print("=" * 70)

# ------------------------------------------------------------------
# Create YOLO object from the CBAM YAML
# ------------------------------------------------------------------
smoke_model = YOLO("/kaggle/working/yolov8n_cbam.yaml")

# ------------------------------------------------------------------
# Verify architecture BEFORE training
# ------------------------------------------------------------------
model_before = smoke_model.model

cbam_before = sum(
    1 for m in model_before.modules()
    if m.__class__.__name__ == "CBAM"
)

c2fcbam_before = sum(
    1 for m in model_before.modules()
    if m.__class__.__name__ == "C2fCBAM"
)

params_before = sum(
    p.numel() for p in model_before.parameters()
)

print("\nArchitecture BEFORE training:")
print(f"CBAM modules    : {cbam_before}")
print(f"C2fCBAM modules : {c2fcbam_before}")
print(f"Parameters      : {params_before:,}")

assert cbam_before == 3
assert c2fcbam_before == 3
assert params_before == 3_023_844

# ------------------------------------------------------------------
# Transfer pretrained YOLOv8n weights again
# ------------------------------------------------------------------
src_state = YOLO("yolov8n.pt").model.state_dict()
dst_state = model_before.state_dict()

compatible = {}

# Direct matches
for key, value in src_state.items():
    if key in dst_state and dst_state[key].shape == value.shape:
        compatible[key] = value

# Transfer original C2f weights into C2fCBAM.c2f
for layer_idx in [4, 6, 8]:
    original_prefix = f"model.{layer_idx}."
    cbam_prefix = f"model.{layer_idx}.c2f."

    for key, value in src_state.items():
        if key.startswith(original_prefix):
            suffix = key[len(original_prefix):]
            new_key = cbam_prefix + suffix

            if (
                new_key in dst_state
                and dst_state[new_key].shape == value.shape
            ):
                compatible[new_key] = value

missing, unexpected = model_before.load_state_dict(
    compatible,
    strict=False
)

print("\nPretrained transfer:")
print(f"Transferred tensors : {len(compatible)}")
print(f"Missing tensors     : {len(missing)}")
print(f"Unexpected tensors  : {len(unexpected)}")

# ------------------------------------------------------------------
# Train for ONE epoch
# ------------------------------------------------------------------
print("\nStarting 1-epoch smoke test...\n")

results = smoke_model.train(
    data="/kaggle/working/visdrone_final.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,

    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    cos_lr=True,
    warmup_epochs=3,

    amp=True,

    seed=42,
    deterministic=True,

    patience=1,
    max_det=1000,

    project="/kaggle/working/YOLO_VisDrone_CBAM_SMOKE",
    name="YOLOv8n_CBAM_1epoch",
    exist_ok=True,

    pretrained=False,
    verbose=True
)

print("\n" + "=" * 70)
print("SMOKE TEST TRAINING COMPLETED")
print("=" * 70)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

print("=" * 70)
print("VERIFYING CBAM CHECKPOINT")
print("=" * 70)

run_dir = Path(
    "/kaggle/working/YOLO_VisDrone_CBAM_SMOKE/YOLOv8n_CBAM_1epoch"
)

best_path = run_dir / "weights" / "best.pt"
last_path = run_dir / "weights" / "last.pt"

print(f"Best checkpoint : {best_path}")
print(f"Exists          : {best_path.exists()}")

print(f"Last checkpoint : {last_path}")
print(f"Exists          : {last_path.exists()}")

assert best_path.exists(), "best.pt was not created."

# Reload the checkpoint
reloaded = YOLO(str(best_path))
model = reloaded.model

cbam_count = sum(
    1 for m in model.modules()
    if m.__class__.__name__ == "CBAM"
)

c2fcbam_count = sum(
    1 for m in model.modules()
    if m.__class__.__name__ == "C2fCBAM"
)

params = sum(
    p.numel() for p in model.parameters()
)

print("\nReloaded checkpoint architecture:")
print(f"CBAM modules    : {cbam_count}")
print(f"C2fCBAM modules : {c2fcbam_count}")
print(f"Parameters      : {params:,}")

assert cbam_count == 3
assert c2fcbam_count == 3
assert params == 3_023_844

print("\n✓ best.pt exists.")
print("✓ Checkpoint reload successful.")
print("✓ 3 CBAM modules preserved.")
print("✓ 3 C2fCBAM modules preserved.")
print("✓ Parameter count preserved.")
print("\n✓ CBAM training pipeline is validated.")

Yes. The smoke test is now sufficiently validated to move to the full benchmark training. I would keep the smoke-test checkpoint separate and not modify the CBAM architecture before the benchmark.
1. YOLOv8n-CBAM Smoke-Test Validation Report
Model
Model: YOLOv8n-CBAM
Dataset: VisDrone
Image size: 640 × 640
Batch size: 8
Epochs: 1
Seed: 42
Optimizer: SGD
Ultralytics: 8.4.203
GPU: NVIDIA Tesla T4 14,912 MiB
Architecture validation
The smoke test reported:
- CBAM modules: 3
- C2fCBAM modules: 3
- Parameters: 3,023,844
- GFLOPs: 8.2
- Layers: 153
The pretrained YOLOv8n weights were successfully transferred:
- Transferred tensors: 319
- Missing tensors: 45
- Unexpected tensors: 0
The model therefore successfully initialized from the pretrained checkpoint.    Pasted markdown(20261106-104722)
Training validation
The training completed successfully:
- Training images: 6,471
- Validation images: 548
- Training batches: 809
- Validation instances: 38,759
- Epochs completed: 1/1
- best.pt generated
- last.pt generated
The training losses decreased during the epoch, confirming that forward propagation, loss computation, backpropagation, and optimizer updates were functioning.    Pasted markdown(20261106-104722)
Validation results
After one epoch:
Metric	Result
Precision	0.0144
Recall	0.117
mAP@50	0.00932
mAP@50–95	0.00267


These values are not suitable for judging the quality of CBAM, because only one epoch was trained. The purpose of this run was pipeline validation, not performance evaluation.    Pasted markdown(20261106-104722)
Checkpoint reload validation
The important part is that best.pt was successfully reloaded and validated.

After model fusion:
- Layers: 96
- Parameters: 3,018,644
- GFLOPs: 8.1
- Validation completed successfully.
The same validation metrics were reproduced after checkpoint reload.    Pasted markdown(20261106-104722)
Speed
The reported inference pipeline speed was approximately:
- Preprocess: 0.1 ms/image
- Inference: 2.2 ms/image
- Postprocess: 2.7 ms/image
   Pasted markdown(20261106-104722)
Final smoke-test conclusion
Status: PASSED

The smoke test confirms that the current YOLOv8n-CBAM pipeline can:

Build → initialize pretrained weights → train → backpropagate → save checkpoint → reload checkpoint → validate → fuse model

Therefore, I recommend not changing the CBAM architecture or training code at this stage.

The next experiment should be the full benchmark training.
2. Move to Full Benchmark Training
For the benchmark, we should now make the experiment controlled.

The most important rule is:
YOLOv8n baseline and YOLOv8n-CBAM must use exactly the same dataset split, image size, batch size, epochs, seed, optimizer, and evaluation procedure.

Otherwise the comparison will be difficult to defend in the paper.

I recommend:
Setting	Benchmark
Dataset	VisDrone
Image size	640
Epochs	100
Batch	8
Seed	42
Optimizer	SGD
Model	YOLOv8n / YOLOv8n-CBAM
Dataset YAML	visdrone_final.yaml
Evaluation	mAP@50, mAP@50–95, Precision, Recall
Complexity	Params, GFLOPs
Speed	Inference time / FPS
Checkpoint	best.pt


Since your earlier run completed 100 epochs in about 2.449 hours, 100 epochs is a reasonable benchmark duration.

In [ ]:
# ============================================================
# CELL  — VERIFY SAVED CBAM CHECKPOINT
# ============================================================

from ultralytics import YOLO
import torch
import os

BEST_PATH = (
    "/kaggle/working/YOLO_VisDrone_CBAM_SMOKE/"
    "YOLOv8n_CBAM_1epoch/weights/best.pt"
)

LAST_PATH = (
    "/kaggle/working/YOLO_VisDrone_CBAM_SMOKE/"
    "YOLOv8n_CBAM_1epoch/weights/last.pt"
)

print("=" * 70)
print("VERIFYING SAVED CBAM CHECKPOINT")
print("=" * 70)

print("\nBest checkpoint :", BEST_PATH)
print("Exists          :", os.path.exists(BEST_PATH))

print("Last checkpoint :", LAST_PATH)
print("Exists          :", os.path.exists(LAST_PATH))


# ------------------------------------------------------------
# Load BEST checkpoint
# ------------------------------------------------------------

verify_model = YOLO(BEST_PATH)

model = verify_model.model


# ------------------------------------------------------------
# Count CBAM modules
# ------------------------------------------------------------

cbam_count = sum(
    1
    for m in model.modules()
    if m.__class__.__name__ == "CBAM"
)

c2fcbam_count = sum(
    1
    for m in model.modules()
    if m.__class__.__name__ == "C2fCBAM"
)


# ------------------------------------------------------------
# Count parameters
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("\nReloaded checkpoint architecture:")
print("CBAM modules      :", cbam_count)
print("C2fCBAM modules   :", c2fcbam_count)
print("Total parameters  :", f"{total_params:,}")
print("Trainable params  :", f"{trainable_params:,}")


# ------------------------------------------------------------
# Expected
# ------------------------------------------------------------

print("\nExpected:")
print("CBAM modules      : 3")
print("C2fCBAM modules   : 3")
print("Total parameters  : 3,023,844")


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert cbam_count == 3, (
    f"ERROR: Expected 3 CBAM modules, found {cbam_count}"
)

assert c2fcbam_count == 3, (
    f"ERROR: Expected 3 C2fCBAM modules, found {c2fcbam_count}"
)

assert total_params == 3_023_844, (
    f"ERROR: Expected 3,023,844 parameters, found {total_params}"
)

print("\n" + "=" * 70)
print("✓ SAVED CBAM CHECKPOINT IS VALID")
print("=" * 70)

In [ ]:
# ============================================================
# FINAL YOLOv8n + CBAM BENCHMARK
# FRESH CBAM ARCHITECTURE + MANUAL YOLOv8n WEIGHT TRANSFER
# NO model.load() -- preserves 10-class CBAM architecture
# ============================================================

import os
import json
import torch
from pathlib import Path
from ultralytics import YOLO

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
YAML_PATH = "/kaggle/working/yolov8n_cbam.yaml"
PRETRAINED_PATH = "/kaggle/working/yolov8n.pt"

PROJECT_DIR = "/kaggle/working/YOLO_VisDrone_Benchmark"
RUN_NAME = "YOLOv8n_CBAM_AdamW_seed42"

RUN_DIR = Path(PROJECT_DIR) / RUN_NAME

print("=" * 70)
print("FINAL YOLOv8n + CBAM BENCHMARK")
print("=" * 70)

print(f"YAML        : {YAML_PATH}")
print(f"Pretrained  : {PRETRAINED_PATH}")
print(f"Project     : {PROJECT_DIR}")
print(f"Run name    : {RUN_NAME}")

assert os.path.exists(YAML_PATH), f"YAML not found: {YAML_PATH}"
assert os.path.exists(PRETRAINED_PATH), \
    f"Pretrained YOLOv8n weights not found: {PRETRAINED_PATH}"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def count_cbam_modules(model):
    return sum(
        1 for m in model.model.modules()
        if m.__class__.__name__ == "CBAM"
    )


def count_c2fcbam_modules(model):
    return sum(
        1 for m in model.model.modules()
        if m.__class__.__name__ == "C2fCBAM"
    )


def count_parameters(model):
    return sum(
        p.numel() for p in model.model.parameters()
    )


# ============================================================
# 1. FRESH CBAM MODEL
# ============================================================

print("\n[1/5] Constructing FRESH CBAM model...")

model = YOLO(YAML_PATH)

cbam_count = count_cbam_modules(model)
c2fcbam_count = count_c2fcbam_modules(model)
params_before = count_parameters(model)

print("\nFresh architecture:")
print(f"  CBAM modules       : {cbam_count}")
print(f"  C2fCBAM modules    : {c2fcbam_count}")
print(f"  Total parameters   : {params_before:,}")

assert cbam_count == 3
assert c2fcbam_count == 3
assert params_before == 3_023_844

print("✓ Fresh CBAM architecture verified")


# ============================================================
# 2. LOAD ORIGINAL YOLOv8n ONLY AS A SOURCE MODEL
# ============================================================

print("\n[2/5] Loading original YOLOv8n source model...")

source = YOLO(PRETRAINED_PATH)

source_state = source.model.state_dict()
target_state = model.model.state_dict()

print(f"Source tensors : {len(source_state)}")
print(f"Target tensors : {len(target_state)}")


# ============================================================
# 3. MANUAL WEIGHT TRANSFER
#
# IMPORTANT:
# - Do NOT call model.load()
# - Do NOT replace model.model
# - Preserve CBAM model architecture
# - Transfer only compatible YOLOv8n weights
#
# C2fCBAM contains the original C2f layer internally:
#
#     model.4.c2f....
#
# whereas YOLOv8n has:
#
#     model.4....
#
# Therefore we explicitly map:
#
#     model.X.<weight>
#       ->
#     model.X.c2f.<weight>
# ============================================================

print("\n[3/5] Transferring compatible YOLOv8n weights manually...")

transferred = 0
skipped = 0
transferred_keys = []
skipped_keys = []

for source_key, source_tensor in source_state.items():

    candidate_keys = []

    # --------------------------------------------------------
    # Direct match
    # --------------------------------------------------------
    candidate_keys.append(source_key)

    # --------------------------------------------------------
    # C2fCBAM wrapped C2f layers
    #
    # model.4.xxx -> model.4.c2f.xxx
    # model.6.xxx -> model.6.c2f.xxx
    # model.8.xxx -> model.8.c2f.xxx
    # --------------------------------------------------------
    parts = source_key.split(".")

    if len(parts) >= 3 and parts[0] == "model":
        try:
            layer_idx = int(parts[1])

            if layer_idx in [4, 6, 8]:
                wrapped_key = (
                    f"model.{layer_idx}.c2f."
                    + ".".join(parts[2:])
                )
                candidate_keys.append(wrapped_key)

        except ValueError:
            pass

    matched = False

    for target_key in candidate_keys:

        if target_key not in target_state:
            continue

        if target_state[target_key].shape != source_tensor.shape:
            continue

        # Transfer tensor
        target_state[target_key].copy_(source_tensor)

        transferred += 1
        transferred_keys.append(
            (source_key, target_key)
        )

        matched = True
        break

    if not matched:
        skipped += 1
        skipped_keys.append(source_key)


# Apply the modified state dictionary WITHOUT replacing architecture
model.model.load_state_dict(target_state, strict=False)


print("\nWeight-transfer summary:")
print(f"  Transferred tensors : {transferred}")
print(f"  Skipped tensors     : {skipped}")

print("\n✓ Compatible YOLOv8n weights transferred")
print("✓ CBAM architecture preserved")
print("✓ Detection head remains 10-class")


# ============================================================
# 4. VERIFY ARCHITECTURE AFTER WEIGHT TRANSFER
# ============================================================

print("\n[4/5] Verifying architecture AFTER weight transfer...")

cbam_after = count_cbam_modules(model)
c2fcbam_after = count_c2fcbam_modules(model)
params_after = count_parameters(model)

print("\nArchitecture after manual transfer:")
print(f"  CBAM modules       : {cbam_after}")
print(f"  C2fCBAM modules    : {c2fcbam_after}")
print(f"  Total parameters   : {params_after:,}")

assert cbam_after == 3, \
    f"Expected 3 CBAM modules, found {cbam_after}"

assert c2fcbam_after == 3, \
    f"Expected 3 C2fCBAM modules, found {c2fcbam_after}"

assert params_after == 3_023_844, \
    f"Architecture changed! Expected 3,023,844 but found {params_after:,}"

print("\n" + "=" * 70)
print("✓ FINAL ARCHITECTURE VERIFIED")
print("=" * 70)
print("  CBAM modules       : 3")
print("  C2fCBAM modules    : 3")
print("  Parameters         : 3,023,844")
print("  Detection classes  : 10")
print("  YOLOv8n weights    : transferred")
print("  Architecture       : preserved")
print("=" * 70)


# ============================================================
# 5. FINAL TRAINING CONFIGURATION
# ============================================================

TRAIN_CONFIG = {
    "data": "/kaggle/working/visdrone_final.yaml",

    "epochs": 100,
    "imgsz": 640,
    "batch": 8,
    "device": 0,

    "optimizer": "AdamW",
    "lr0": 0.002,
    "weight_decay": 0.0005,

    "cos_lr": True,
    "warmup_epochs": 3,

    "amp": True,

    "seed": 42,
    "deterministic": True,

    "patience": 20,
    "max_det": 1000,

    "workers": 2,

    "copy_paste": 0.0,

    # IMPORTANT:
    # We already transferred YOLOv8n weights manually.
    "pretrained": False,

    "project": PROJECT_DIR,
    "name": RUN_NAME,

    "exist_ok": False,

    "verbose": True
}


# ============================================================
# 6. START FRESH 100-EPOCH RUN
# ============================================================

print("\n[5/5] Starting FINAL 100-epoch CBAM benchmark...")

print("\nTraining configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key:16s}: {value}")

print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)

results = model.train(**TRAIN_CONFIG)


# ============================================================
# 7. CHECKPOINT VERIFICATION
# ============================================================

BEST_PATH = RUN_DIR / "weights" / "best.pt"

print("\n" + "=" * 70)
print("CBAM TRAINING COMPLETED")
print("=" * 70)

print(f"Best checkpoint : {BEST_PATH}")
print(f"Exists          : {BEST_PATH.exists()}")

assert BEST_PATH.exists(), \
    "best.pt was not created."

print("✓ best.pt successfully created")


# ============================================================
# 8. SAVE BENCHMARK RECORD
# ============================================================

metadata = {
    "experiment": "YOLOv8n + CBAM",
    "dataset": "VisDrone_Final",

    "architecture": {
        "base_model": "YOLOv8n",
        "attention": "CBAM",
        "cbam_modules": 3,
        "c2fcbam_modules": 3,
        "total_parameters": 3_023_844,
        "num_classes": 10
    },

    "initialization": {
        "method": "fresh model from yolov8n_cbam.yaml",
        "pretrained_source": "original YOLOv8n pretrained weights",
        "transfer_method": "manual compatible state-dict transfer",
        "resume": False
    },

    "weight_transfer": {
        "transferred_tensors": transferred,
        "skipped_tensors": skipped
    },

    "training": TRAIN_CONFIG,

    "paths": {
        "best_pt": str(BEST_PATH),
        "run_dir": str(RUN_DIR),
        "results_csv": str(RUN_DIR / "results.csv")
    }
}

metadata_path = RUN_DIR / "benchmark_record.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nBenchmark record:")
print(metadata_path)

print("\n" + "=" * 70)
print("FINAL CBAM BENCHMARK READY")
print("=" * 70)

In [8]:
from ultralytics import YOLO
runs = {  # name: path to best.pt
    "YOLOv8n_CBAM": "/kaggle/working/YOLO_VisDrone_Benchmark/YOLOv8n_CBAM_AdamW_seed42/weights/best.pt",
    # add the other models here
}
for name, w in runs.items():
    YOLO(w).val(data="visdrone_final.yaml", split="test", imgsz=640,
                conf=0.001, iou=0.7, max_det=500, save_json=True, name=name)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n_cbam summary (fused): 96 layers, 3,018,644 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 371.0±140.9 MB/s, size: 217.8 KB)
val: Scanning /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 708.6it/s 2.3s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 4.0it/s 25.4s0.2s
                   all       1610      75102      0.348      0.273      0.233      0.126
            pedestrian       1197      21006       0.33      0.237      0.185     0.0663
                people        797       6376      0.449      0.093     0.0998     0.029

In [18]:
# ============================================================
# SIZE-STRATIFIED AP / RECALL EVALUATION
# ============================================================

!pip install -q pycocotools pillow

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


BINS = {
    "coco": [
        ("small", 0, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],

    "fine": [
        ("tiny", 0, 16**2),
        ("small", 16**2, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],
}


def build_gt(img_dir, lbl_dir, n_classes, cat_offset):

    images = []
    anns = []
    aid = 1

    for p in sorted(Path(img_dir).iterdir()):

        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        w, h = Image.open(p).size

        iid = int(p.stem) if p.stem.isnumeric() else p.stem

        images.append(
            dict(
                id=iid,
                width=w,
                height=h,
                file_name=p.name
            )
        )

        lf = Path(lbl_dir) / (p.stem + ".txt")

        if not lf.exists():
            continue

        for line in lf.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) < 5:
                continue

            c = int(float(parts[0]))

            xc, yc, bw, bh = (
                float(v) for v in parts[1:5]
            )

            # Convert normalized YOLO coordinates
            # to ORIGINAL image pixel coordinates
            bw = bw * w
            bh = bh * h

            x = xc * w - bw / 2
            y = yc * h - bh / 2

            anns.append(
                dict(
                    id=aid,
                    image_id=iid,
                    category_id=c + cat_offset,
                    bbox=[x, y, bw, bh],
                    area=bw * bh,
                    iscrowd=0
                )
            )

            aid += 1

    cats = [
        dict(
            id=i + cat_offset,
            name=str(i)
        )
        for i in range(n_classes)
    ]

    gt = COCO()

    gt.dataset = dict(
        images=images,
        annotations=anns,
        categories=cats
    )

    gt.createIndex()

    return gt


def mean_valid(a):

    a = a[a > -1]

    return float(a.mean()) if a.size else float("nan")


def evaluate(gt, pred_file, bins, maxdet):

    with open(pred_file, "r") as f:
        preds = json.load(f)

    dt = gt.loadRes(preds)

    E = COCOeval(
        gt,
        dt,
        "bbox"
    )

    E.params.maxDets = [10, 100, maxdet]

    E.params.areaRng = (
        [[0, 1e10]]
        + [[lo, hi] for _, lo, hi in bins]
    )

    E.params.areaRngLbl = (
        ["all"]
        + [b[0] for b in bins]
    )

    E.evaluate()
    E.accumulate()

    prec = E.eval["precision"]
    rec = E.eval["recall"]

    # maxDet index
    m = 2

    rows = []

    areas = np.array([
        a["area"]
        for a in gt.dataset["annotations"]
    ])

    for a, lbl in enumerate(E.params.areaRngLbl):

        lo, hi = E.params.areaRng[a]

        n = int(
            ((areas >= lo) & (areas <= hi)).sum()
        )

        rows.append(
            dict(
                bin=lbl,
                n_gt=n,

                AP50_95=mean_valid(
                    prec[:, :, :, a, m]
                ),

                AP50=mean_valid(
                    prec[0, :, :, a, m]
                ),

                AR50_95=mean_valid(
                    rec[:, :, a, m]
                ),

                AR50=mean_valid(
                    rec[0, :, a, m]
                )
            )
        )

    return rows


def main(argv=None):

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--images",
        required=True
    )

    ap.add_argument(
        "--labels",
        required=True
    )

    ap.add_argument(
        "--runs",
        nargs="+",
        required=True,
        help="NAME=path/to/predictions.json"
    )

    ap.add_argument(
        "--nc",
        type=int,
        default=10
    )

    ap.add_argument(
        "--bins",
        choices=BINS,
        default="fine"
    )

    ap.add_argument(
        "--maxdet",
        type=int,
        default=500
    )

    ap.add_argument(
        "--cat_offset",
        type=int,
        default=-1,
        help="0 or 1; -1 = infer from predictions"
    )

    ap.add_argument(
        "--out",
        default="size_results.csv"
    )

    a = ap.parse_args(argv)

    runs = dict(
        r.split("=", 1)
        for r in a.runs
    )

    # --------------------------------------------------------
    # Infer category offset
    # --------------------------------------------------------

    off = a.cat_offset

    if off < 0:

        first_file = next(
            iter(runs.values())
        )

        with open(first_file, "r") as f:
            first = json.load(f)

        off = min(
            p["category_id"]
            for p in first
        )

        print(
            "Inferred category offset:",
            off
        )

    # --------------------------------------------------------
    # Build ground truth
    # --------------------------------------------------------

    gt = build_gt(
        a.images,
        a.labels,
        a.nc,
        off
    )

    print(
        f"GT: {len(gt.dataset['images'])} images, "
        f"{len(gt.dataset['annotations'])} boxes"
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    out = []

    for name, f in runs.items():

        print()
        print("=" * 70)
        print(f"Evaluating: {name}")
        print(f"Prediction file: {f}")
        print("=" * 70)

        for r in evaluate(
            gt,
            f,
            BINS[a.bins],
            a.maxdet
        ):

            r["model"] = name

            out.append(r)

            print(
                f"{name:12s} "
                f"{r['bin']:7s} "
                f"n={r['n_gt']:6d} "
                f"AP50={r['AP50']:.3f} "
                f"AP50-95={r['AP50_95']:.3f} "
                f"AR50={r['AR50']:.3f} "
                f"AR50-95={r['AR50_95']:.3f}"
            )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    with open(
        a.out,
        "w",
        newline=""
    ) as fh:

        w = csv.DictWriter(
            fh,
            fieldnames=[
                "model",
                "bin",
                "n_gt",
                "AP50",
                "AP50_95",
                "AR50",
                "AR50_95"
            ]
        )

        w.writeheader()
        w.writerows(out)

    print()
    print("=" * 70)
    print("Saved:")
    print(a.out)
    print("=" * 70)


# ============================================================
# KAGGLE CONFIGURATION
# ============================================================

main([
    "--images",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/images",

    "--labels",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels",

    "--runs",

    "CBAM=/kaggle/working/runs/detect/YOLOv8n_CBAM/predictions.json",

    # "YOLOv9t=/kaggle/working/runs/detect/v9t/predictions.json",

    "--nc",
    "10",

    "--bins",
    "fine",

    "--maxdet",
    "500",

    "--cat_offset",
    "-1",

    "--out",
    "/kaggle/working/size_results.csv"
])

Inferred category offset: 1
creating index...
index created!
GT: 1610 images, 75102 boxes

Evaluating: CBAM
Prediction file: /kaggle/working/runs/detect/YOLOv8n_CBAM/predictions.json
Loading and preparing results...
DONE (t=4.67s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=225.68s).
Accumulating evaluation results...
DONE (t=14.46s).
CBAM         all     n= 75102 AP50=0.236 AP50-95=0.129 AR50=0.541 AR50-95=0.289
CBAM         tiny    n= 27346 AP50=0.070 AP50-95=0.025 AR50=0.246 AR50-95=0.092
CBAM         small   n= 23491 AP50=0.188 AP50-95=0.078 AR50=0.544 AR50-95=0.247
CBAM         medium  n= 21856 AP50=0.356 AP50-95=0.196 AR50=0.748 AR50-95=0.424
CBAM         large   n=  2409 AP50=0.476 AP50-95=0.317 AR50=0.832 AR50-95=0.566

Saved:
/kaggle/working/size_results.csv
